<a href="https://colab.research.google.com/github/gjvlio/emotion-based-multimodal-deepfake-detector/blob/feat%2Ftraining-turnover-prep/training-turnover-prep/notebooks/2_Train_And_Evaluate_DeepSentinel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepSentinel: End-to-End Training & 10,000-Clip Benchmark Evaluation

This notebook runs the **Free Google Colab (T4 GPU)** training pipeline and evaluates out-of-domain performance on **FakeAVCeleb**.

### **Pipeline Stages**:
1. **Stage 1 (Head Pre-training)**: Pre-trains the 299-D multi-scale fusion head, SE-Attention block, and GELU activations (~60 seconds).
2. **Stage 2 (Backbone Fine-tuning)**: Fine-tunes Wav2Vec2/ViT backbones with FP16, `pos_weight = 1.3835`, LayerNorm, and differential learning rates (~45 minutes).
3. **FakeAVCeleb Benchmark**: Evaluates on 10,000 cached FakeAVCeleb clips with Youden's J Statistic sweep (~30 seconds).

In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 1: Fetch clean updated code (Commit: e96b315)
%cd /content
import os
if not os.path.exists("/content/thesis"):
    !git clone -b feat/training-turnover-prep https://github.com/gjvlio/emotion-based-multimodal-deepfake-detector.git /content/thesis

%cd /content/thesis
!git fetch origin feat/training-turnover-prep
!git reset --hard e96b315595c244b496b5e8bac630d8547df2a099
!pip install -q transformers scikit-learn tensorboard timm pandas openai-whisper opencv-python-headless


/content
Cloning into '/content/thesis'...
remote: Enumerating objects: 1075, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 1075 (delta 154), reused 138 (delta 85), pack-reused 834 (from 1)
Receiving objects: 100% (1075/1075), 6.07 MiB | 16.19 MiB/s, done.
Resolving deltas: 100% (636/636), done.
/content/thesis
From https://github.com/gjvlio/emotion-based-multimodal-deepfake-detector
 * branch            feat/training-turnover-prep -> FETCH_HEAD
HEAD is now at e96b315 fix(drive): ensure immediate os.sync() and multi-location backup to Google Drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# Step 3: Run Stage 1 Head Pre-training (~60 seconds)
%cd /content/thesis
!python scripts/colab_stage1.py

/content/thesis

  DEEP-SENTINEL COLAB: STAGE 1 TRAINING (EMOTION BILINEAR HEAD)

Scanning Google Drive (THESIS_MOTHERFILE) for zip archives...
  Found zip: datasets/cmumosei.zip (6635.0 MB)
  Found zip: datasets/crema_d_raw.zip (2955.6 MB)
  Found zip: datasets/tracks_1_2_3_4.zip (13054.9 MB)
  Found zip: datasets/meld_raw.zip (11803.6 MB)
  Found zip: datasets/mustard.zip (1558.8 MB)
  Found zip: datasets/fakeavceleb.zip (6407.6 MB)
  Found zip: preprocessed/preprocessed.zip (2910.7 MB)
  Found zip: preprocessed/CMUMOSEI/metadata.zip (0.7 MB)
  Found zip: preprocessed/CMUMOSEI/mosei_features_shard2.zip (17.4 MB)
  Found zip: preprocessed/CMUMOSEI/mosei_features_shard3.zip (15.1 MB)
  Found zip: preprocessed/CMUMOSEI/mosei_features_shard0.zip (15.0 MB)
  Found zip: preprocessed/CMUMOSEI/mosei_features_shard1.zip (15.0 MB)

  PRE-FLIGHT DATASET & FEATURE CACHE VALIDATOR

Checking and extracting all feature archives from Google Drive...
  Extracting preprocessed.zip (2910.7 MB) from pre

In [ ]:
# Cell 3: Fine-Tune Top-4 Backbone Layers with Live E2E Val & Multi-Location Drive Flush (~2.5 hrs)
%cd /content/thesis
!python scripts/colab_stage2.py

/content/thesis

  DEEP-SENTINEL COLAB: STAGE 2 TRAINING (BACKBONE FINE-TUNING)

Scanning Google Drive (THESIS_MOTHERFILE) for zip archives...
  Found zip: datasets/cmumosei.zip (6635.0 MB)
  Found zip: datasets/crema_d_raw.zip (2955.6 MB)
  Found zip: datasets/tracks_1_2_3_4.zip (13054.9 MB)
  Found zip: datasets/meld_raw.zip (11803.6 MB)
  Found zip: datasets/mustard.zip (1558.8 MB)
  Found zip: datasets/fakeavceleb.zip (6407.6 MB)
  Found zip: preprocessed/preprocessed.zip (2910.7 MB)
  Found zip: preprocessed/CMUMOSEI/metadata.zip (0.7 MB)
  Found zip: preprocessed/CMUMOSEI/mosei_features_shard2.zip (17.4 MB)
  Found zip: preprocessed/CMUMOSEI/mosei_features_shard3.zip (15.1 MB)
  Found zip: preprocessed/CMUMOSEI/mosei_features_shard0.zip (15.0 MB)
  Found zip: preprocessed/CMUMOSEI/mosei_features_shard1.zip (15.0 MB)

  PRE-FLIGHT DATASET & FEATURE CACHE VALIDATOR (STAGE 2)

Checking and extracting all feature archives from Google Drive...
  [ALREADY EXTRACTED] Skipping preprocess

In [ ]:
# Cell 4: Calibrated 500 Real / 500 Fake End-to-End Benchmark Evaluation (~30 mins)
%cd /content/thesis
!python scripts/colab_eval_fakeav.py --n_real 500 --n_fake 500